## Validação Manual do PSI
Este notebook calcula o PSI mensalmente (dev e oot) em relação ao global de dev, sem usar a biblioteca do ModelSentry.

In [1]:
import pandas as pd
import numpy as np
import sqlite3

# Conectar ao banco simulado
conn = sqlite3.connect('../data/simulacao.db')
df = pd.read_sql('SELECT * FROM tb_dados_escorados', conn)
conn.close()

# Mostrar a distribuição de dados por safra
display(df.groupby(['data_ref', 'amostra']).size().reset_index(name='qtd'))


,data_ref,amostra,qtd
0,2022-01-01,dev,300
1,2022-02-01,dev,300
2,2022-03-01,dev,300
3,2022-04-01,dev,300
4,2022-05-01,dev,300
5,2022-06-01,dev,300
6,2022-07-01,dev,300
7,2022-08-01,dev,300
8,2022-09-01,dev,300
9,2022-10-01,dev,300


In [2]:
def calculate_psi(expected, actual, bins=10):
    # Calcula decis usando os dados de referência (usando decis quebra em 10 quantis)
    # Aqui estamos usando a função np.histogram que quebra em 10 intervalos iguais (bins=10).
    # Uma implementação clássica de PSI costuma usar decis (quantis). Vamos testar das duas formas!
    
    # Abordagem 1: Bins iguais (a que estamos usando no momento)
    expected_counts, bin_edges = np.histogram(expected.dropna(), bins=bins, density=False)
    actual_counts, _ = np.histogram(actual.dropna(), bins=bin_edges, density=False)
    
    expected_pct = expected_counts / len(expected)
    actual_pct = actual_counts / len(actual)
    
    # Previne divisão por zero e log(0)
    expected_pct = np.where(expected_pct == 0, 0.0001, expected_pct)
    actual_pct = np.where(actual_pct == 0, 0.0001, actual_pct)
    
    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi * 100 # Em percentual


In [3]:
df_ref = df[df['amostra'] == 'dev']
expected_scores = df_ref['score']

resultados = []
for data_ref, group in df.groupby('data_ref'):
    psi_val = calculate_psi(expected_scores, group['score'])
    amostra_tipo = group['amostra'].iloc[0]
    resultados.append({'data_ref': data_ref, 'amostra': amostra_tipo, 'psi_pct': psi_val})

df_psi = pd.DataFrame(resultados).sort_values('data_ref')
display(df_psi)


,data_ref,amostra,psi_pct
0,2022-01-01,dev,2.010017
1,2022-02-01,dev,1.864789
2,2022-03-01,dev,3.769452
3,2022-04-01,dev,1.472356
4,2022-05-01,dev,1.035325
5,2022-06-01,dev,2.339441
6,2022-07-01,dev,1.675743
7,2022-08-01,dev,1.074734
8,2022-09-01,dev,1.706270
9,2022-10-01,dev,2.921213


In [4]:
# Abordagem 2: Quebrando por Decis Reais (qcut)
# Pode ser que os intervalos iguais do np.histogram estejam deixando bins vazios e inflando o PSI

def calculate_psi_deciles(expected, actual, q=10):
    # Define os decis com base na distribuição de referência
    _, bin_edges = pd.qcut(expected, q=q, retbins=True, duplicates='drop')
    
    # Ajusta os limites para englobar os dados de atual que possam fugir dos extremos
    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf
    
    # Calcula as distribuições
    expected_counts = pd.cut(expected, bins=bin_edges).value_counts(sort=False).values
    actual_counts = pd.cut(actual, bins=bin_edges).value_counts(sort=False).values
    
    expected_pct = expected_counts / len(expected)
    actual_pct = actual_counts / len(actual)
    
    expected_pct = np.where(expected_pct == 0, 0.0001, expected_pct)
    actual_pct = np.where(actual_pct == 0, 0.0001, actual_pct)
    
    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi * 100

resultados_decis = []
for data_ref, group in df.groupby('data_ref'):
    psi_val = calculate_psi_deciles(expected_scores, group['score'])
    amostra_tipo = group['amostra'].iloc[0]
    resultados_decis.append({'data_ref': data_ref, 'amostra': amostra_tipo, 'psi_pct_deciles': psi_val})

df_psi_decis = pd.DataFrame(resultados_decis).sort_values('data_ref')
display(df_psi_decis)


,data_ref,amostra,psi_pct_deciles
0,2022-01-01,dev,0.971786
1,2022-02-01,dev,1.730741
2,2022-03-01,dev,2.093583
3,2022-04-01,dev,4.443999
4,2022-05-01,dev,1.469058
5,2022-06-01,dev,2.683148
6,2022-07-01,dev,1.972839
7,2022-08-01,dev,2.558047
8,2022-09-01,dev,4.332024
9,2022-10-01,dev,2.675711
